In [1]:
import os
from typing import Any
from numpy import dtype, ndarray, signedinteger
from numpy._typing import _32Bit, _64Bit
from qdrant_client.http.models import Distance, VectorParams
from qdrant_client import QdrantClient, models
from langchain_core.documents import Document

In [2]:
import glob
import json

list_file = glob.glob("data/LuatLaoDong2025_temp/*.json")


def load_points():
    points = []
    for filename in list_file:
        print(f"process:  {filename}", end="\n")
        try:
            with open(filename, "r", encoding="utf-8") as f:
                data = json.load(f)

                points.append(
                    models.PointStruct(
                        id=data["id"],
                        payload=data["payload"],
                        vector={
                            "metadata_embedded": data["vector"]["metadata_embedded"],
                            "page_content_embedded": data["vector"]["page_content_embedded"],
                            "fuse_embedded": data["vector"]["fuse_embedded"],
                        }
                    )
                )
            print(f"{filename} loaded", end="\n")
        except Exception as e:
            print(f"failed:  {filename}, error: {e}", end="\n")
    return points


points = load_points()

process:  data/LuatLaoDong2025_temp1\004a6e97-39a6-4e85-b284-04dc662c01f8.json
data/LuatLaoDong2025_temp1\004a6e97-39a6-4e85-b284-04dc662c01f8.json loaded
process:  data/LuatLaoDong2025_temp1\00ad2ced-6916-4ecd-a583-7546a0728da5.json
data/LuatLaoDong2025_temp1\00ad2ced-6916-4ecd-a583-7546a0728da5.json loaded
process:  data/LuatLaoDong2025_temp1\0179a986-81bd-4544-b62f-ee455136867f.json
data/LuatLaoDong2025_temp1\0179a986-81bd-4544-b62f-ee455136867f.json loaded
process:  data/LuatLaoDong2025_temp1\017fafb2-f545-4229-b582-d7166d70e4ad.json
data/LuatLaoDong2025_temp1\017fafb2-f545-4229-b582-d7166d70e4ad.json loaded
process:  data/LuatLaoDong2025_temp1\01964028-16fb-4c14-9f74-cb582cc1b496.json
data/LuatLaoDong2025_temp1\01964028-16fb-4c14-9f74-cb582cc1b496.json loaded
process:  data/LuatLaoDong2025_temp1\02369dc8-dddd-4b44-bdad-8534f57cc16d.json
data/LuatLaoDong2025_temp1\02369dc8-dddd-4b44-bdad-8534f57cc16d.json loaded
process:  data/LuatLaoDong2025_temp1\0238c58f-5770-4a51-9be7-371fc6be1

In [4]:
client = QdrantClient(url="http://localhost:6333")
NAME_COLLECTION = "RAG_lawyer"

In [5]:
client.create_collection(
    collection_name=NAME_COLLECTION,
    vectors_config={
        "metadata_embedded": models.VectorParams(size=1024, distance=models.Distance.COSINE),
        "page_content_embedded": models.VectorParams(size=1024, distance=models.Distance.COSINE),
        "fuse_embedded": models.VectorParams(size=1024, distance=models.Distance.COSINE)
    },
)

True

In [6]:
for i in range(0, len(points), 20):
    client.upsert(
        collection_name=NAME_COLLECTION,
        points=points[i:i + 20],
    )